Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

Visualizing an image

In [ ]:
# load image
image_path = "image.jpg"

img_bgr = cv2.imread(image_path)

if img_bgr is None:
  raise ValueError("Image not found")

# opencv loads as BGR -> convert to RGB
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# normalize to [0, 1]
img_rgb = img_rgb.astype(np.float32) / 255.0

print("Shape: ", img_rgb.shape)

Extraacting the intensity channel

In [ ]:
intensity = np.mean(img_rgb, axis=2)
print("Intensity shape:", intensity.shape)

Visualization of both RGB and intesity map

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(img_rgb)
ax[0].set_title("RGB Image")
ax[0].axis("off")

ax[1].imshow(intensity, cmap='gray')
ax[1].set_title("Intensity Map")
ax[1].axis("off")

plt.tight_layout()
plt.show()

Multi-scale Guassian pyramids

In [ ]:
def build_gaussian_pyramid(image, levels):
  pyramid = [image]
  current = image.copy()

  for _ in range(levels - 1):
    current = cv2.pyrDown(current)
    pyramid.append(current)

  return pyramid

Intensity pyramid

In [ ]:
intensity_pyramid = build_gaussian_pyramid(intensity, levels=9)
print("NUmber of pyramid levels: ", len(intensity_pyramid))

Visualize the pyramid

In [ ]:
fig, axes = plt.subplots(1, len(intensity_pyramid), figsize=(18, 4))

for i, level in enumerate(intensity_pyramid):
    axes[i].imshow(level, cmap='gray')
    axes[i].set_title(f"L{i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

Center surround difference

In [ ]:
def center_surround_diff(pyramid, c, s):
  center = pyramid[c]
  surround = pyramid[s]

  # resize surround to center size
  surround_resized = cv2.resize(
      surround,
      (center.shape[1], center.shape[0]),
      interpolation=cv2.INTER_LINEAR
  )

  diff = cv2.absdiff(center, surround_resized)

  return diff

Generating one feature map

In [ ]:
feature_map = center_surround_diff(
    intensity_pyramid,
    c=2,
    s=5
)

plt.figure(figsize=(6,6))
plt.imshow(feature_map, cmap='gray')
plt.title("Center-Surround Feature Map")
plt.axis("off")
plt.show()

Generate all center-surround feature maps

In [ ]:
feature_maps = []

center_scales = [2, 3, 4]
delta_scales = [3, 4]

for c in center_scales:
  for delta in delta_scales:
    s = c + delta

    fmap = center_surround_diff(
        intensity_pyramid,
        c = c,
        s = s
    )

    feature_maps.append(fmap)

print("Total feature maps: ", len(feature_maps))

Visualize them all

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, fmap, idx in zip(axes.flat, feature_maps, range(len(feature_maps))):
    ax.imshow(fmap, cmap='gray')
    ax.set_title(f"Map {idx+1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def normalize_map(feature_map):
    fmap = feature_map.copy()

    fmap = cv2.normalize(
        fmap,
        None,
        alpha=0,
        beta=1,
        norm_type=cv2.NORM_MINMAX
    )

    mean_val = np.mean(fmap)
    max_val = np.max(fmap)

    fmap = fmap * ((max_val - mean_val) ** 2)

    return fmap

normalize all feature maps

In [ ]:
normalized_maps = [
    normalize_map(fmap)
    for fmap in feature_maps
]

Combining them into intensity conspicuity map

In [ ]:
base_shape = normalized_maps[0].shape

intensity_conspicuity = np.zeros(base_shape, dtype=np.float32)

for fmap in normalized_maps:
    resized = cv2.resize(
        fmap,
        (base_shape[1], base_shape[0])
    )

    intensity_conspicuity += resized

# Final normalization
intensity_conspicuity = cv2.normalize(
    intensity_conspicuity,
    None,
    0,
    1,
    cv2.NORM_MINMAX
)

Visualize it

In [ ]:
plt.figure(figsize=(7,7))

plt.imshow(intensity_conspicuity, cmap='hot')
plt.title("Intensity Conspicuity Map")
plt.axis("off")

plt.colorbar()
plt.show()

Adding pre-processing block

In [ ]:
R = img_rgb[:, :, 0]
G = img_rgb[:, :, 1]
B = img_rgb[:, :, 2]

# Intensity reused from earlier
I = intensity + 1e-6

Compute opponent channel

In [ ]:
r = R - (G + B) / 2
g = G - (R + B) / 2
b = B - (R + G) / 2
y = (R + G) / 2 - np.abs(R - G) / 2 - B

# Remove negatives
r[r < 0] = 0
g[g < 0] = 0
b[b < 0] = 0
y[y < 0] = 0

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

channels = [r, g, b, y]
titles = ["Red", "Green", "Blue", "Yellow"]

for ax, ch, title in zip(axes, channels, titles):
    ax.imshow(ch, cmap='gray')
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

Constructing RG and BY opponency maps

In [ ]:
RG = cv2.absdiff(r, g)
BY = cv2.absdiff(b, y)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(RG, cmap='hot')
axes[0].set_title("Red-Green Opponency")
axes[0].axis("off")

axes[1].imshow(BY, cmap='hot')
axes[1].set_title("Blue-Yellow Opponency")
axes[1].axis("off")

plt.tight_layout()
plt.show()

Guassian pyramids for both channels

In [ ]:
RG_pyramid = build_gaussian_pyramid(RG, levels=9)
BY_pyramid = build_gaussian_pyramid(BY, levels=9)

Feature maps

In [ ]:
color_feature_maps = []

for pyramid in [RG_pyramid, BY_pyramid]:
  for c in center_scales:
    for delta in delta_scales:
      s = c + delta
      fmap = center_surround_diff(
        pyramid,
        c=c,
        s=s
      )
      fmap = normalize_map(fmap)
      color_feature_maps.append(fmap)

Combine into one colour conspicuity map

In [ ]:
base_shape = color_feature_maps[0].shape

color_conspicuity = np.zeros(base_shape, dtype=np.float32)

for fmap in color_feature_maps:

    resized = cv2.resize(
        fmap,
        (base_shape[1], base_shape[0])
    )

    color_conspicuity += resized

color_conspicuity = cv2.normalize(
    color_conspicuity,
    None,
    0,
    1,
    cv2.NORM_MINMAX
)

In [ ]:
plt.figure(figsize=(7,7))

plt.imshow(color_conspicuity, cmap='hot')
plt.title("Color Conspicuity Map")
plt.axis("off")

plt.colorbar()
plt.show()

Reusable Gabor Filtering function

In [ ]:
def apply_gabor_filter(image, theta, ksize=9):

    kernel = cv2.getGaborKernel(
        (ksize, ksize),
        sigma=4.0,
        theta=theta,
        lambd=10.0,
        gamma=0.5,
        psi=0,
        ktype=cv2.CV_32F
    )

    filtered = cv2.filter2D(
        image,
        cv2.CV_32F,
        kernel
    )

    return filtered

Define orientations

In [ ]:
orientations = [
    0,
    np.pi / 4,
    np.pi / 2,
    3 * np.pi / 4
]

Applying filters to the intensity image

In [ ]:
orientation_maps = []

for theta in orientations:

    response = apply_gabor_filter(
        intensity,
        theta
    )

    orientation_maps.append(response)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

titles = [
    "0°",
    "45°",
    "90°",
    "135°"
]

for ax, omap, title in zip(axes, orientation_maps, titles):

    ax.imshow(omap, cmap='gray')
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

Building pyramid for each orientation

In [ ]:
orientation_pyramids = []

for omap in orientation_maps:

    pyramid = build_gaussian_pyramid(
        omap,
        levels=9
    )

    orientation_pyramids.append(pyramid)

Generate center surround feature maps

In [ ]:
orientation_feature_maps = []

for pyramid in orientation_pyramids:

    for c in center_scales:
        for delta in delta_scales:

            s = c + delta

            fmap = center_surround_diff(
                pyramid,
                c=c,
                s=s
            )

            fmap = normalize_map(fmap)

            orientation_feature_maps.append(fmap)

Combine these into one orientation conspicuity map

In [ ]:
base_shape = orientation_feature_maps[0].shape

orientation_conspicuity = np.zeros(
    base_shape,
    dtype=np.float32
)

for fmap in orientation_feature_maps:

    resized = cv2.resize(
        fmap,
        (base_shape[1], base_shape[0])
    )

    orientation_conspicuity += resized

orientation_conspicuity = cv2.normalize(
    orientation_conspicuity,
    None,
    0,
    1,
    cv2.NORM_MINMAX
)

In [ ]:
plt.figure(figsize=(7,7))

plt.imshow(
    orientation_conspicuity,
    cmap='hot'
)

plt.title("Orientation Conspicuity Map")
plt.axis("off")

plt.colorbar()
plt.show()

Normalize all conspicuity maps again before fusion

In [ ]:
I_bar = normalize_map(intensity_conspicuity)
C_bar = normalize_map(color_conspicuity)
O_bar = normalize_map(orientation_conspicuity)

Fuse them equally

In [ ]:
saliency_map = (
    I_bar +
    C_bar +
    O_bar
) / 3.0

saliency_map = cv2.normalize(
    saliency_map,
    None,
    0,
    1,
    cv2.NORM_MINMAX
)

Visualize the raw saliency map

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(
    saliency_map,
    cmap='hot'
)

plt.title("Final Saliency Map")
plt.axis("off")

plt.colorbar()
plt.show()

Convert saliency to heatmap

In [ ]:
# Resize saliency map to original image size
saliency_resized = cv2.resize(
    saliency_map,
    (img_rgb.shape[1], img_rgb.shape[0])
)

# Convert to uint8
saliency_uint8 = np.uint8(
    saliency_resized * 255
)

# Generate heatmap
heatmap = cv2.applyColorMap(
    saliency_uint8,
    cv2.COLORMAP_JET
)

# Convert BGR -> RGB
heatmap = cv2.cvtColor(
    heatmap,
    cv2.COLOR_BGR2RGB
)

Overlay it to original

In [ ]:
overlay = cv2.addWeighted(
    np.uint8(img_rgb * 255),
    0.6,
    heatmap,
    0.4,
    0
)

Visualize everything together

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(saliency_map, cmap='hot')
axes[1].set_title("Saliency Map")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title("Attention Heatmap Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()